In [ ]:
import numpy as np
import adaptive_latents
import matplotlib.pyplot as plt
import scipy

rng = np.random.default_rng()

In [ ]:
thetas = np.linspace(0, 2 * np.pi, 101)[:-1]

X_true = np.array([scipy.stats.vonmises(kappa=1,loc=theta).pdf(thetas) for theta in thetas])
X = X_true + 0.02 * rng.standard_normal(X_true.shape)
plt.matshow(X)
plt.colorbar()

In [ ]:
p = adaptive_latents.Pipeline([
    adaptive_latents.CenteringEstimator(),
    pro:=adaptive_latents.proSVD(k=2),
])

for _ in range(1):
    p.offline_run_on(X_true + 0.02 * rng.standard_normal(X_true.shape))

p.freeze()
latents = p.offline_run_on(X)


In [ ]:
plt.plot(latents[:,0], latents[:,1])
plt.plot(latents[-1,0], latents[-1,1], 'oC0')
plt.axis('equal')


In [ ]:
sd = adaptive_latents.stim_designer.StimDesigner(lam_1=0)
stim = sd.design_stim(np.array([0,1]).reshape((-1,1)), u_dimension=100, u_to_s_function=lambda x: pro.Q.T @ x)
plt.plot(stim)
plt.plot(X[-1])

In [ ]:
from sklearn.cross_decomposition import PLSRegression

reference = np.vstack((np.cos(thetas), np.sin(thetas))).T

neural_data = X_true + .05 * rng.standard_normal(X_true.shape)

reg = PLSRegression(n_components=2).fit(neural_data, reference)


In [ ]:
propls = adaptive_latents.proPLS(k=2)


propls.offline_run_on([neural_data, reference])

In [ ]:
plt.plot(propls.u)
plt.title('proPLS loadings')

In [ ]:
d = adaptive_latents.datasets.Zong22Dataset(2)

In [ ]:
from adaptive_latents import ArrayWithTime, Pipeline, CenteringEstimator, KernelSmoother

reference = np.vstack([np.sin(d.behavior_df.hd), np.cos(d.behavior_df.hd)]).T
reference = ArrayWithTime(reference, t=d.behavior_df.t + 4*d.neural_data.dt)

p = Pipeline([
    CenteringEstimator(init_size=100),
    KernelSmoother(tau=8),
    propls:=adaptive_latents.proPLS(k=2)
])
p.offline_run_on([d.neural_data, reference])

p.steps[0].freeze()
p.steps[1] = p.steps[1].blank_copy()
p.steps[2].freeze()

latents = p.offline_run_on([d.neural_data, reference])


In [ ]:
fig, ax = plt.subplots()
ax.plot(d.neural_data.t, d.neural_data[:,0])
ks = KernelSmoother(tau=8)
smoothed = ks.offline_run_on(d.neural_data)
ax.plot(smoothed.t -  4*d.neural_data.dt, smoothed[:,0])
ax.set_xlim(0,20)

In [ ]:
fig, ax = plt.subplots()
d.show_stim_pattern(ax, desired_stim=np.linalg.norm(propls.u, axis=1))

In [ ]:
hd = adaptive_latents.utils.resample_matched_timeseries(d.behavior_df.hd.to_numpy()[:,None], d.behavior_df.t.to_numpy(), d.neural_data.t)


idx = np.argsort(np.linalg.norm(propls.u, axis=1))[-2]

fig, axs = plt.subplots(nrows=2, figsize=(5,10))
axs[0].plot(d.neural_data.t, d.neural_data[:,idx])
axs[0].plot(d.neural_data.t, hd)

axs[1].plot(hd, d.neural_data[:,idx], '.k', markersize=1)
axs[1].set_xlabel('head direction (rad)')
axs[1].set_ylabel('df/f')

